# Morphological Transformations

## Objectives
- Learn fundamental morphological operations: erosion, dilation, opening, closing, and hit-or-miss transform
- Explore advanced morphological operations based on morphological reconstruction
- Apply morphological operations to grayscale images: erosion, dilation, opening, closing, Top-Hat, and Bottom-Hat filters
- Use compound morphological operations to solve a practical image-processing problem
- Homework: implement Conway's Game of Life using morphology

## Theory Overview

### Structuring Element

A structuring element is a small image patch (in discrete image representation, a subset of pixels). The most common choices are 3×3 or 5×5 squares. Other shapes, such as approximations of an ellipse, are also used when needed.

### Erosion

Erosion is a fundamental morphological operation. Assume the input image contains a region (foreground) X distinguished by some characteristic feature (e.g., brightness relative to the background). After erosion, X becomes the set of center points of all structuring elements that fit entirely inside X. The strength of erosion is controlled by the size of the structuring element.

Erosion can be viewed as a minimum filter: for each pixel neighborhood defined by the structuring element, the minimum value is assigned to the output image.

### Dilation

Dilation: Assume the input image contains a region X distinguished by some characteristic feature (e.g., brightness). After dilation, X becomes the set of center points of all structuring elements whose at least one point lies inside X. The strength of dilation is controlled by the size of the structuring element.

Dilation can be viewed as a maximum filter: for each pixel neighborhood defined by the structuring element, the maximum value is assigned to the output image.

### Opening and Closing

Opening consists of erosion followed by dilation.

> Opening = erosion + dilation

Closing consists of dilation followed by erosion.

> Closing = dilation + erosion

### Grayscale Images

Grayscale images — valley and peak detection (Top-Hat, Bottom-Hat):

Local extrema can be extracted using opening and closing. To detect local maxima (peaks), subtract the original image from its opening. To detect local minima, perform the analogous operation with closing as the first step. Note: these methods detect and highlight local extrema only.

## Basic Morphological Operations: Erosion, Dilation, Opening, Closing, Hit-or-Miss

1. Load the image `ertka.bmp`.
2. Apply erosion with `cv2.erode`. The function takes the image and a structuring element. The element can be created manually as a 0/1 array (`np.ones((3,3))`) or with `cv2.getStructuringElement`, passing a shape such as `cv2.MORPH_RECT` and size `(3,3)`. Start with a 3-pixel square.
3. Display the original image and the eroded result side by side. Verify that you understand how erosion works.
4. Change the structuring element (different shape — circle, diamond — or size). Apply erosion again and inspect the result.
5. Besides changing the structuring element, erosion strength can be increased by running multiple iterations (e.g., three passes). Keep a 3×3 square element. Erode `ertka` twice, then three times, and compare the results. Hint: see the documentation for `erode`.
6. Load `buzka.bmp`. Design a structuring element manually as a 0/1 matrix to remove hair strokes of a chosen orientation (diagonal left or right).
7. Note: the same ways of tuning erosion apply to dilation, opening, and closing.
8. Dilation (`cv2.dilate`) is the dual operation to erosion. Use a 3×3 square structuring element and dilate `ertka`. Inspect the result.
9. On one figure, show the original image and the results of erosion, dilation, opening, and closing. Opening and closing are obtained with `cv2.morphologyEx(img, operation, structuring_element)`, using `cv2.MORPH_OPEN` or `cv2.MORPH_CLOSE`.
10. Repeat the four operations on `wyspa.bmp` and `kolka.bmp`. Compare the outcomes.
11. Mini-task: using the operations above, keep only the letters RT in `ertka.bmp` (without protrusions or holes).
12. To detect specific pixel configurations, use the hit-or-miss transform. It finds groups of pixels that exactly match a structuring element pattern.
13. Load `hom.bmp` and display it. Assume we want to detect 3×3 cross patterns. Define the following structuring element:
```
[0,1,0]
[1,1,1]
[0,1,0]
```
Apply hit-or-miss: `cv2.morphologyEx(hom, cv2.MORPH_HITMISS, se1)`. Display the result. Did the task succeed? If you get data-type errors, convert the input to grayscale: `cv2.cvtColor(hom, cv2.COLOR_BGR2GRAY)`.


In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
import os

if not os.path.exists("buzka.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/10_Morphology/buzka.bmp --no-check-certificate
if not os.path.exists("calculator.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/10_Morphology/calculator.bmp --no-check-certificate
if not os.path.exists("ertka.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/10_Morphology/ertka.bmp --no-check-certificate
if not os.path.exists("ferrari.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/10_Morphology/ferrari.bmp --no-check-certificate
if not os.path.exists("fingerprint.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/10_Morphology/fingerprint.bmp --no-check-certificate
if not os.path.exists("hom.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/10_Morphology/hom.bmp --no-check-certificate
if not os.path.exists("kolka.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/10_Morphology/kolka.bmp --no-check-certificate
if not os.path.exists("kosc.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/10_Morphology/kosc.bmp --no-check-certificate
if not os.path.exists("szkielet.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/10_Morphology/szkielet.bmp --no-check-certificate
if not os.path.exists("text.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/10_Morphology/text.bmp --no-check-certificate
if not os.path.exists("wyspa.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/10_Morphology/wyspa.bmp --no-check-certificate
if not os.path.exists("rice.png") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/10_Morphology/rice.png --no-check-certificate

# TODO
def s(*img, titles=None):
    f, ax = plt.subplots(1, len(img), figsize=(4 * len(img), 4))
    if len(img) == 1:
        ax = [ax]
    for i, j in enumerate(img):
        ax[i].imshow(j, cmap='gray')
        ax[i].axis('off')
        if titles is not None:
            ax[i].set_title(titles[i])
    plt.show()

def rec(mrk, msk, k=None):
    if k is None:
        k = np.ones((3, 3), np.uint8)
    m = mrk.copy()
    while True:
        tmp = np.minimum(cv2.dilate(m, k), msk)
        if np.array_equal(m, tmp):
            return m
        m = tmp

e = cv2.imread('ertka.bmp', cv2.IMREAD_GRAYSCALE)
k3 = np.ones((3, 3), np.uint8)
k5e = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

s(e, cv2.erode(e, k3))
s(e, cv2.erode(e, k5e))
s(cv2.erode(e, k3, iterations=2), cv2.erode(e, k3, iterations=3))

b = cv2.imread('buzka.bmp', cv2.IMREAD_GRAYSCALE)
se_b = np.array([[1, 0, 0],
                 [0, 1, 0],
                 [0, 0, 1]], np.uint8)
s(b, cv2.erode(b, se_b))

s(e, cv2.dilate(e, k3))

for n in ['ertka.bmp', 'wyspa.bmp', 'kolka.bmp']:
    i = cv2.imread(n, cv2.IMREAD_GRAYSCALE)
    s(i,
      cv2.erode(i, k3),
      cv2.dilate(i, k3),
      cv2.morphologyEx(i, cv2.MORPH_OPEN, k3),
      cv2.morphologyEx(i, cv2.MORPH_CLOSE, k3))

rt = cv2.morphologyEx(e, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
rt = cv2.morphologyEx(rt, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
s(e, rt)

h = cv2.imread('hom.bmp', cv2.IMREAD_GRAYSCALE)
_, hbin = cv2.threshold(h, 0, 1, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
se1 = np.array([[0, 1, 0],
                [1, 1, 1],
                [0, 1, 0]], np.int8)
hhm = cv2.morphologyEx(hbin, cv2.MORPH_HITMISS, se1)
s(hbin * 255, hhm * 255)

t = cv2.imread('text.bmp', cv2.IMREAD_GRAYSCALE)
pion = np.ones((51, 1), np.uint8)
m = cv2.erode(t, pion)
r = rec(m, t)
s(t, cv2.morphologyEx(t, cv2.MORPH_OPEN, pion), r)

f = cv2.imread('ferrari.bmp', cv2.IMREAD_GRAYSCALE)
k3 = np.ones((3, 3), np.uint8)
er = cv2.erode(f, k3)
dl = cv2.dilate(f, k3)
s(f, er, dl, cv2.subtract(dl, er))
s(f,
  cv2.morphologyEx(f, cv2.MORPH_OPEN, k3),
  cv2.morphologyEx(f, cv2.MORPH_CLOSE, k3))
s(f,
  cv2.morphologyEx(f, cv2.MORPH_TOPHAT, k3),
  cv2.morphologyEx(f, cv2.MORPH_BLACKHAT, k3))

rc = cv2.imread('rice.png', cv2.IMREAD_GRAYSCALE)
s(rc, cv2.morphologyEx(rc, cv2.MORPH_TOPHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))))

c = cv2.imread('calculator.bmp', cv2.IMREAD_GRAYSCALE)
p71 = np.ones((1, 71), np.uint8)

r1 = rec(cv2.erode(c, p71), c)
s(c, r1, cv2.morphologyEx(c, cv2.MORPH_OPEN, p71))

th_r = cv2.subtract(c, r1)
s(th_r, cv2.morphologyEx(c, cv2.MORPH_TOPHAT, p71))

r2 = rec(cv2.erode(th_r, np.ones((1, 11), np.uint8)), th_r)
s(r2)

r3 = rec(np.minimum(cv2.dilate(r2, np.ones((1, 21), np.uint8)), th_r), th_r)
s(r3)

## Additional Morphological Operations
Other morphological operations include thinning, skeletonization, morphological reconstruction, border clearing, and hole filling. This section focuses on morphological reconstruction.

Morphological reconstruction is a three-argument operation. It requires a marker (the image from which the transform starts), a mask (the constraint on the transform), and a structuring element. The procedure repeats until two consecutive iterations are identical:
- dilate the marker with the given structuring element,
- new marker = intersection of the dilated marker and the mask.

Three operations that use this reconstruction scheme are:
- opening by reconstruction,
- hole filling,
- border clearing.

### Opening by Reconstruction
- Load `text.bmp` and display it.
- Assume we want to detect letters containing long vertical strokes. As a first approach, apply morphological opening with a vertical structuring element of height 51 pixels (the average letter height on the image — `np.ones((51,1))`). Inspect the result.
- Detection works, but only vertical strokes remain.
- Reconstruction solves this: use the eroded original image as the marker and the original image as the mask. Choose the structuring element yourself.
- Implement reconstruction and compare opening with reconstruction.


In [ ]:
t = cv2.imread('text.bmp', 0)
pion = np.ones((51,1), np.uint8)

def rec(mrk, msk):
    m = mrk.copy()
    k = np.ones((3,3), np.uint8)
    while True:
        tmp = np.minimum(cv2.dilate(m, k), msk)
        if np.array_equal(m, tmp): return m
        m = tmp

s(t, cv2.morphologyEx(t, cv2.MORPH_OPEN, pion), rec(cv2.erode(t, pion), t))

## Morphological Operations on Grayscale Images

All operations covered so far (except hit-or-miss) have grayscale counterparts. Erosion and dilation are defined as:
- Erosion — minimum filter.
- Dilation — maximum filter.


1. Load `ferrari.bmp` and apply erosion and dilation with a 3×3 square structuring element. Compute the morphological gradient as the difference between the dilated and eroded images. Display all results side by side.
2. Opening suppresses bright details; closing suppresses dark details. Verify this on `ferrari`.
3. Apply Top-Hat and Bottom-Hat with `cv2.morphologyEx(img, cv2.MORPH_TOPHAT, strel)` and `cv2.morphologyEx(img, cv2.MORPH_BLACKHAT, strel)` on `ferrari`. Which regions are highlighted? What operations compose the Top-Hat filter?
4. Load `rice.png` (from the binarization lab) and display it. Note the uneven illumination. Apply Top-Hat with a large structuring element (e.g., a circle of size 10). Display the result. What happened to the illumination non-uniformity?

In [ ]:
f = cv2.imread('ferrari.bmp', 0)
k3 = np.ones((3,3), np.uint8)
er = cv2.erode(f, k3)
dl = cv2.dilate(f, k3)
s(f, er, dl, cv2.subtract(dl, er))

s(f, cv2.morphologyEx(f, cv2.MORPH_OPEN, k3), cv2.morphologyEx(f, cv2.MORPH_CLOSE, k3))

s(f, cv2.morphologyEx(f, cv2.MORPH_TOPHAT, k3), cv2.morphologyEx(f, cv2.MORPH_BLACKHAT, k3))

rc = cv2.imread('rice.png', 0)
s(rc, cv2.morphologyEx(rc, cv2.MORPH_TOPHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21,21))))

## Morphology Application Example

1. Load `calculator.bmp` and display it. Goal: isolate the text on the calculator keys.
2. First, remove horizontal reflections along the top edge of each key. The reflection is longer than any single character. Use opening by reconstruction (reuse code from the previous exercise; this time the image is grayscale rather than binary — consider which operation corresponds to logical AND):
  - start with erosion using a horizontal line structuring element — `np.ones((1,71))`,
  - then reconstruct: marker = eroded image, mask = original image,
  - display the result. For comparison, show classical opening with the same structuring element. Why is opening by reconstruction preferable?
3. The previous step yields a background estimate. Subtract it from the original image. This is reconstruction-based Top-Hat. Display the result and compare it with classical Top-Hat — the difference between the original and the classically opened image.
4. Remove vertical glare in the same way:
  - erosion with a horizontal line structuring element — `np.ones((1,11))` — preserves most characters (nearly all are wider). Apply this to the result from step 3 (original minus background).
  - reconstruction: marker = eroded image, mask = image from step 3,
  - display the result.
5. The result is nearly satisfactory, but thin vertical strokes (e.g., the I in ASIN) are lost. Because removed strokes lie next to surviving ones, apply:
  - dilation with `np.ones((1,21))`,
  - reconstruction with marker = minimum(dilated image from above, image from step 3) and mask = image from step 3.
6. Display the final result. Did the proposed pipeline achieve the intended text extraction?


In [ ]:
c = cv2.imread('calculator.bmp', 0)
p71 = np.ones((1,71), np.uint8)
r1 = rec(cv2.erode(c, p71), c)
s(c, r1, cv2.morphologyEx(c, cv2.MORPH_OPEN, p71))

th_r = cv2.subtract(c, r1)
s(th_r, cv2.morphologyEx(c, cv2.MORPH_TOPHAT, p71))

r2 = rec(cv2.erode(th_r, np.ones((1,11), np.uint8)), th_r)
s(r2)

r3 = rec(np.minimum(cv2.dilate(r2, np.ones((1,21), np.uint8)), th_r), th_r)
s(r3)